In [3]:
# Run this cell FIRST
!pip install transformers datasets accelerate -q

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from datasets import load_dataset, DatasetDict, Audio
import torch

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# In your training loop / Trainer:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/cv_ar_model",
    save_steps=500,           # save every 500 steps
    save_total_limit=2,       # keep only last 2 checkpoints
    resume_from_checkpoint=True,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'TrainingArguments' is not defined

In [ ]:
!git clone 'https://github.com/jumon/whisper-finetuning.git'
%cd whisper-finetuning

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!apt-get install ffmpeg -y

In [ ]:
!pip install -U openai-whisper

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip uninstall -y transformers tokenizers

In [ ]:
!pip install -U transformers tokenizers

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -U datasets

In [ ]:
import whisper
model = whisper.load_model("base")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p data/audio
!mkdir -p data/transcripts

In [ ]:
pip install datacollective

In [ ]:
from datasets import load_dataset

dataset = load_dataset("librispeech_asr", "clean", streaming=True)

In [ ]:
print(dataset)

In [ ]:
!pip install datasets soundfile librosa -q


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset
import soundfile as sf
import numpy as np
import os

os.makedirs("data/audio", exist_ok=True)
os.makedirs("data/trans", exist_ok=True)

# Option A — Classical Arabic, ~9700 samples, 12 hours (recommended)
ds = load_dataset("MBZUAI/ClArTTS", split="train")

# Option B — smaller, ~1900 samples
# ds = load_dataset("tunis-ai/arabic_speech_corpus", split="train")

print(f"Total samples: {len(ds)}")
print("Columns:", ds.column_names)

In [ ]:
sample = ds[0]
for col in ds.column_names:
    val = sample[col]
    print(f"{col}: type={type(val).__name__} → {str(val)[:200]}")

In [ ]:
import soundfile as sf
import numpy as np
import os

for i, sample in enumerate(ds):
    stem = f"ar_{i:05d}"

    audio_array = np.array(sample["audio"], dtype=np.float32)
    sample_rate  = sample["sampling_rate"]  # 40100

    sf.write(f"data/audio/{stem}.wav", audio_array, sample_rate)

    with open(f"data/transcripts/{stem}.txt", "w", encoding="utf-8") as f:
        f.write(sample["text"].strip())

    if i % 500 == 0:
        print(f"  {i}/{len(ds)}")

print("Done!")

In [ ]:
!echo "Audio:"; ls data/audio | wc -l
!echo "Trans:"; ls data/transcripts | wc -l
!echo "Sample transcript:"; cat data/trans/ar_00000.txt

In [ ]:
import os

with open("data/data.tsv", "w", encoding="utf-8") as out:
    for fname in sorted(os.listdir("data/audio")):

        if not fname.endswith(".wav"):
            continue

        stem = os.path.splitext(fname)[0]

        t_path = f"data/transcripts/{stem}.txt"

        if not os.path.exists(t_path):
            print("Missing transcript:", t_path)
            continue

        with open(t_path, encoding="utf-8") as t:
            text = t.read().strip()

        out.write(f"data/audio/{fname}\t{text}\n")

print("Saved data/data.tsv")

In [ ]:
# split 90% train, 10% dev
import os

lines = open("data/data.tsv", encoding="utf-8").readlines()
split = int(len(lines) * 0.9)

with open("data/train.tsv", "w", encoding="utf-8") as f:
    f.writelines(lines[:split])

with open("data/dev.tsv", "w", encoding="utf-8") as f:
    f.writelines(lines[split:])

print(f"Train: {split} samples")
print(f"Dev:   {len(lines)-split} samples")

In [ ]:
!python create_data.py \
    --without-timestamps \
    --data-file data/train.tsv \
    --language ar \
    --output data/train.json

!python create_data.py \
    --without-timestamps \
    --data-file data/dev.tsv \
    --language ar \
    --output data/dev.json

In [ ]:
! pip install bitsandbytes==0.41.1

In [ ]:
!python run_finetuning.py \
    --train-json data/train.json \
    --dev-json   data/dev.json \
    --model      small \
    --use-adam-8bit \
    --train-steps     3000 \
    --eval-steps       300 \
    --batch-size         1 \
    --accum-grad-steps  16 \
    --warmup-steps      200 \
    --lr             1e-4 \
    --save-dir       output/